In [ ]:
#| hide
from laghu import *

# laghu

> model2vec static models in a fraction of the memory, with the same vectors

laghu (Sanskrit for light) loads a [model2vec](https://github.com/MinishLab/model2vec) static embedding model and gives back an object
with model2vec's `encode`, `encode_as_sequence` and `tokenize`. The output is model2vec's to the bit. What changes is where the model
lives.

model2vec reads the embedding table into each process. For potion-multilingual-128M that is 512 MB, and the Unigram tokenizer adds a
character trie over its 500,353 pieces, another 610 MB or so: about 1.3 GB of private memory in every worker that encodes. laghu maps
the table read-only from `model.safetensors`, so the operating system keeps one copy in the page cache for every process that maps it,
and can drop those pages under memory pressure because they are clean. For a Unigram model it also replaces the trie with a Viterbi
pass in Python over a dict of the pieces. The normalizer and pre-tokenizer still come from `tokenizers`, built from the same
tokenizer.json with the vocabulary left out. The same model then takes about 135 MB a process.

WordPiece, BPE and word-level models are tokenized by `tokenizers` as before. Their vocabularies are small hash maps, so the saving
there is the table alone, which is most of the memory for every model in the table below.

## Install

Not on PyPI yet. From a checkout:

```sh
uv add ../laghu
```

model2vec itself is not a dependency. For the fallback described below, and for `same_as_model2vec`, install the extra:
`uv add '../laghu[fallback]'`.

## Use

In [ ]:
#| eval: false
from laghu import load

m = load('minishlab/potion-multilingual-128M')      # a folder, or a Hugging Face repo id (the local cache is tried first)
v = m.encode(['The quick brown fox.', 'कर्मण्येवाधिकारस्ते'])   # float32, (2, 256), unit rows: the same array model2vec returns
m.tokenize(['hello world'])
m.encode_as_sequence('hello world').shape          # (n_tokens, 256)

`load` returns a `LeanModel`. If laghu cannot read the model for any reason, it logs a warning saying why and returns model2vec's own
`StaticModel` instead, so a caller always gets an encoder. `load(..., fallback=False)` raises instead. A bfloat16 or float8 table is
refused with the dtype named: numpy cannot hold either, and model2vec cannot load them either.

`same_as_model2vec(path)` loads a model both ways and compares them on the built-in corpus, or on texts of your own:

In [ ]:
#| eval: false
from laghu import same_as_model2vec
same_as_model2vec('minishlab/potion-base-8M', ['my', 'own texts'])

It checks the token ids, the vectors cast to float32 and to float16, the exact dtype and bits, `encode_as_sequence`,
`median_token_length`, the unknown token id and the token list.

## What matches, and what it costs

Every model below gives model2vec's ids and vectors, bit for bit, on 4,726 texts: prose in 36 languages, random strings from 30
Unicode blocks, and strings picked to break tokenizers (control characters, the special tokens themselves, emoji sequences, combining
marks, bidi controls, 200,000 characters). Private memory is measured after encoding, each in a fresh process; peak is the most private
memory the process held. Speed is `encode` over 3,226 texts on an M3 Max.

| model | tokenizer | laghu tokenizes with | vocab × dims | dtype | norm | identical | private MB, model2vec → laghu | peak MB, model2vec → laghu | texts/s, model2vec → laghu |
|---|---|---|---|---|---|---|---|---|---|
| potion-base-2M | WordPiece | tokenizers | 29,528 × 64 | float32 | on | yes | 81 → 58 | 82 → 58 | 84,510 → 92,982 |
| potion-base-4M | WordPiece | tokenizers | 29,528 × 128 | float32 | on | yes | 94 → 62 | 95 → 62 | 87,248 → 90,242 |
| potion-base-8M | WordPiece | tokenizers | 29,528 × 256 | float32 | on | yes | 114 → 70 | 115 → 70 | 79,415 → 81,810 |
| potion-base-32M | WordPiece | tokenizers | 63,091 × 512 | float32 | on | yes | 244 → 97 | 244 → 97 | 75,009 → 81,379 |
| potion-retrieval-32M | WordPiece | tokenizers | 63,091 × 512 | float32 | on | yes | 244 → 97 | 244 → 97 | 70,799 → 79,717 |
| potion-science-8M | WordPiece | tokenizers | 29,528 × 256 | float32 | on | yes | 113 → 70 | 114 → 70 | 81,008 → 84,532 |
| potion-code-16M | WordPiece | tokenizers | 61,826 × 256 | float32 +map +weights | on | yes | 147 → 110 | 211 → 117 | 10,674 → 10,425 |
| potion-code-16M-v2 | WordPiece | tokenizers | 63,457 × 256 | float16 | on | yes | 108 → 101 | 186 → 119 | 21,819 → 22,164 |
| potion-multilingual-128M | Unigram | Python Viterbi | 500,353 × 256 | float32 | on | yes | 1,348 → 135 | 1,348 → 135 | 39,105 → 12,795 |
| M2V_base_output | WordPiece | tokenizers | 29,528 × 256 | float32 | off | yes | 112 → 69 | 113 → 69 | 65,460 → 87,100 |
| M2V_multilingual_output | WordPiece | tokenizers | 501,054 × 256 | float32 | off | yes | 834 → 198 | 842 → 295 | 42,565 → 90,283 |
| M2V_base_glove | WordLevel | tokenizers | 400,002 × 256 | float32 | off | yes | 656 → 186 | 656 → 194 | 67,028 → 85,058 |
| M2V_base_glove_subword | WordPiece | tokenizers | 400,779 × 256 | float32 | off | yes | 647 → 178 | 649 → 195 | 66,256 → 85,430 |
| jina-embeddings-v3-separation-distilled | Unigram | Python Viterbi | 250,002 × 256 | float32 | off | yes | 722 → 88 | 722 → 88 | 51,836 → 20,840 |
| potion-mxbai-micro | WordPiece | tokenizers | 2,000 × 256 | int8 +map +weights | on | yes | 90 → 72 | 91 → 72 | 68,385 → 70,998 |

On a Unigram model laghu encodes at a third to half of model2vec's rate: `tokenizers` encodes a batch on every core, and the Python
Viterbi runs on one. For a server that encodes a query at a time this does not show; for a bulk pass over a corpus it does.

## What is not supported

- Quantizing or truncating on load (`quantize_to`, `dimensionality`, `vocabulary_quantization`): each would copy the table, which is
  what laghu exists to avoid. Load with model2vec for those.
- `single_word` added tokens on a Unigram model fall back to `tokenizers` for tokenizing (the table is still mapped). Rust decides a
  word boundary by Unicode's Alphabetic property, which Python's `str.isalpha` does not match.
- `use_multiprocessing` is accepted and ignored; the result is the same.
- `encode([])` returns an empty `(0, dim)` array. model2vec raises.

## Developing

nbdev 3 on hatchling: the notebooks in `nbs/` are the source. `uv run nbdev-prepare` exports, tests and rebuilds this README;
`uv run nbdev-test --flags slow` also runs the bench, which downloads about 5 GB of models.

## Licence

MIT.